# Methodology Review — Exhibits

Companion to `METHODOLOGY.md` (the design-decision tour of Knotek & Zaman 2014).
One exhibit per stop; each cell saves its figure to `figs/` for the .md to embed.
Data comes from the kit's `data/` cache (run the skeleton's M9 refresh cell to update).

In [ ]:
# --- setup: cached data + the two tiny transforms the exhibits need ----------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def load(code):
    return pd.read_csv(f"data/{code}.csv", index_col=0, parse_dates=True).iloc[:, 0]

LVL = {"cpi": load("CPIAUCSL"), "core_cpi": load("CPILFESL"),
       "food": load("CPIUFDSL"), "gas_cpi": load("CUSR0000SETB01"),
       "pce": load("PCEPI"), "core_pce": load("PCEPILFE"),
       "gas_wk": load("GASALLW"), "oil": load("DCOILBRENTEU")}

def monthly_inflation(levels):
    return 100 * (levels / levels.shift(1) - 1)

def weekly_to_monthly(weekly):
    m = weekly.groupby(weekly.index.to_period("M")).mean()
    m.index = m.index.to_timestamp()
    return m

print("loaded:", ", ".join(f"{k} → {s.index[-1].date()}" for k, s in list(LVL.items())[:3]), "...")

In [ ]:
# --- exhibit 0: the whole pipeline on one page --------------------------------
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

def box(ax, x, y, w, h, title, lines, fc):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.06",
                                fc=fc, ec="gray", lw=1))
    ax.text(x + w/2, y + h - 0.25, title, ha="center", va="top",
            fontsize=8.5, fontweight="bold")
    ax.text(x + w/2, y + h - 0.62, "\n".join(lines), ha="center", va="top", fontsize=7)

fig, ax = plt.subplots(figsize=(13, 5.5))
ax.set_xlim(0, 13); ax.set_ylim(0, 6); ax.axis("off")

box(ax, 0.1, 3.2, 1.8, 2.4, "① DATA (8 series)",
    ["Brent oil (daily)", "EIA gas (weekly, NSA)", "CPI family (SA,", "out mid t+1)",
     "PCE family", "(out end t+1)"], "#eef4fb")
box(ax, 2.3, 3.2, 1.8, 2.4, "② TRANSFORMS",
    ["daily/weekly → monthly", "NSA gas → SA via", "CPI-gas factors (fn7)",
     "levels → MoM inflation"], "#eef4fb")
box(ax, 4.5, 3.2, 2.3, 2.4, "③ COMPONENTS",
    ["core & food: recursive", "12-month MA (eq 4)", "core PCE bridge (eq 5)",
     "gas: partial-month avg,", "else oil-ECM (eq 6–7),", "random-walk oil (fn9)"], "#eefbf0")
box(ax, 7.2, 3.2, 1.9, 2.4, "④ SWITCHING",
    ["information set s(t)", "→ coefficients switch", "deterministically (eq 3)",
     "paper: 6 cases/month", "kit: A · B · C"], "#fdf6e3")
box(ax, 9.5, 3.2, 2.0, 2.4, "⑤ HEADLINE",
    ["CPI out → bridge (eq 9)", "gas in hand → OLS on", "core+food+gas hats",
     "(eq 10, zero restr.)", "else 12m MA (eq 11)"], "#fdeeee")
box(ax, 2.3, 0.4, 2.3, 1.9, "⑥ MONTHLY OUTPUT",
    ["nowcasts of CPI,", "core CPI, PCE,", "core PCE for month t"], "#f3eefb")
box(ax, 5.2, 0.4, 2.6, 1.9, "⑦ QUARTERLY (eq 1–2)",
    ["quarterly level = mean of", "3 monthly levels; inflation", "= (ratio)^4 − 1 (BEA conv.)",
     "carryover: T−1 months are", "already in T's base"], "#f3eefb")
box(ax, 8.4, 0.4, 2.9, 1.9, "⑧ EVALUATION",
    ["real-time ALFRED vintages", "RMSE by case (declines!)", "DM tests (Harvey)",
     "vs Blue Chip, SPF,", "Greenbook"], "#f7f7f7")

for x0, x1 in [(1.9, 2.3), (4.1, 4.5), (6.8, 7.2), (9.1, 9.5)]:
    ax.add_patch(FancyArrowPatch((x0, 4.4), (x1, 4.4), arrowstyle="-|>",
                                 mutation_scale=14, color="gray"))
ax.add_patch(FancyArrowPatch((10.5, 3.2), (3.4, 2.3), arrowstyle="-|>",
                             mutation_scale=14, color="gray",
                             connectionstyle="arc3,rad=0.25"))
for x0, x1 in [(4.6, 5.2), (7.8, 8.4)]:
    ax.add_patch(FancyArrowPatch((x0, 1.35), (x1, 1.35), arrowstyle="-|>",
                                 mutation_scale=14, color="gray"))

ax.set_title("Knotek–Zaman (2014): the full nowcasting pipeline", fontsize=11)
plt.tight_layout()
plt.savefig("figs/00_pipeline.png", dpi=130, bbox_inches="tight")
plt.show()